In [ ]:
import requests
import pandas as pd
import time
from datetime import timezone, datetime
import os

def get_all_open_interest(symbol="BTCUSDT", category="linear", interval="30min", max_requests=500):
    """
    :param symbol: ticker
    :param interval: '5min', '15min', '30min', '1h', '4h', '1d'
    """
    url = "https://api.bybit.com/v5/market/open-interest"
    
    params = {
        "category": category,
        "symbol": symbol.upper(),
        "intervalTime": interval,
        "limit": 200 
    }
    
    all_data = []
    cursor = None
    request_count = 0
    
    print(f"Start fetching {symbol} ({interval})...")
    
    while request_count < max_requests:
        if cursor:
            params["cursor"] = cursor
            
        response = requests.get(url, params=params)
        data = response.json()
        request_count += 1
        
        if data.get("retCode") != 0:
            print(f"API error #{request_count}: {data.get('retMsg')}")
            break
            
        result = data.get("result", {})
        items = result.get("list", [])
        
        if not items:
            break

        all_data.extend(items)
        cursor = result.get("nextPageCursor")
        
        if request_count % 10 == 0:
            print(f"Loaded bars {len(all_data)} (request #{request_count})")
            
        if not cursor:
            break
            
        time.sleep(0.05)
    else:
        print(f"Iterations limit ({max_requests}).")

    if not all_data:
        print("No data")
        return pd.DataFrame()
        
    df = pd.DataFrame(all_data)
    
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='ms', utc=True)
    df['oi_total'] = df['openInterest'].astype(float)
    df['singleOpenInterest'] = df['singleOpenInterest'].astype(float)
    
    # Chronological sort
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    df_final = df[['timestamp', 'oi_total', 'singleOpenInterest']]
    
    # Data
    start_dt = df_final['timestamp'].iloc[0].strftime('%Y-%m-%d %H:%M')
    end_dt = df_final['timestamp'].iloc[-1].strftime('%Y-%m-%d %H:%M')
    
    print(f"Load completed")
    print(f"Total bars {len(df_final)}")
    print(f"Period from {start_dt} to {end_dt}")
    print(f"Total API requests {request_count}")
    
    return df_final



# USAGE 

if __name__ == "__main__":
    SYMBOL = "BTCUSDT" # Instrument
    INTERVAL = "30min" #Timeframe 
    
    df_full = get_all_open_interest(symbol=SYMBOL, interval=INTERVAL)
    
    if not df_full.empty:
        start = df_full['timestamp'].iloc[0].strftime('%Y%m%d')
        end = df_full['timestamp'].iloc[-1].strftime('%Y%m%d')
        
        filename = f"data/{SYMBOL}_oi_{INTERVAL}_FULL_{start}_to_{end}.csv"
        
        os.makedirs("data", exist_ok=True)
        df_full.to_csv(filename, index=False)
        print(f"File saved: {filename}")